# FE10 Extensions：NULL / Negative-Control 校准

这个 Notebook 只负责读取、展示和解释 NULL 校准结果，不在 Notebook 内训练模型。

正式 Reference 固定为 **FE10 — `unknown_activity_time`**。所有变化都使用同一 Fold 下的 `delta_vs_reference`：

```text
AUC(FE10 + NULL feature) - AUC(FE10)
```

## 实验设计

| Experiment | Control | Folds | 作用 |
|---|---|---:|---|
| `NULL_A_AFFINE` | `2 * daily_screen_time_hours - 3` | 5 | 严格冗余列，只增加 split competition |
| `NULL_B_PERM_1001` | permutation，seed 1001 | 5 | 保留边际分布与缺失比例，打乱行级关系 |
| `NULL_B_PERM_1002` | permutation，seed 1002 | 5 | 第二个独立 permutation |

共 15 个 fits。运行命令（从项目根目录执行）：

```bash
python -m src.noise_calibration
```

脚本支持断点续跑；已有结果只有在协议、配置、Reference、Fold 和线程数全部匹配时才会跳过。

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results" / "noise_calibration"
RESULTS_DIR

## 逐折结果

In [ ]:
records = []
for path in sorted(RESULTS_DIR.glob("null_*_fold*.json")):
    records.append(json.loads(path.read_text(encoding="utf-8")))

fold_results = pd.DataFrame(records)

if fold_results.empty:
    print("尚未找到 NULL 结果。请先从项目根目录运行：python -m src.noise_calibration")
else:
    display(
        fold_results[[
            "experiment_id",
            "fold",
            "reference_auc",
            "auc",
            "delta_vs_reference",
            "best_iteration",
            "elapsed_seconds",
        ]].sort_values(["experiment_id", "fold"])
    )

## 汇总指标

In [ ]:
summary_path = RESULTS_DIR / "summary.csv"

if summary_path.exists():
    null_summary = pd.read_csv(summary_path)
    display(null_summary)
else:
    print("summary.csv 尚未生成。")

## Paired delta 可视化

In [ ]:
if not fold_results.empty:
    pivot = fold_results.pivot(
        index="fold",
        columns="experiment_id",
        values="delta_vs_reference",
    )
    display(pivot)

    ax = pivot.plot(marker="o", figsize=(10, 5))
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("NULL controls: paired AUC delta vs FE10")
    ax.set_xlabel("Fold")
    ax.set_ylabel("AUC delta")
    ax.grid(alpha=0.25)
    plt.show()

## 判读边界

NULL 的用途是校准 feature-addition noise，而不是构造正式 p-value。正式候选仍需同时检查：

- 5-Fold `mean_delta` 是否稳定为正；
- 其量级是否明显区别于 NULL 的 `mean_abs_delta` 和 `std_delta`；
- Fold 方向是否一致；
- `best_iteration` 是否出现异常漂移。

在 NULL 完成前，可以实现下一阶段的特征和 runner；但不要据此决定 group 是否成功、是否进入 LOO，或者是否唤醒 Conditional Reserve。